# HIVE organ training (single GPU)

This notebook trains the one whole-gland organ model from the current deployed `model.ts`. It uses the full HIVE inference `output/` tree as the SGH source, never `Interesting outputs/`.

Safety and data policy:

- Every training container is restricted to physical GPU 0 and must see exactly one CUDA device.
- A lock prevents this organ notebook and the lesion notebook from training concurrently on GPU 0.
- All HIVE outputs are scanned, but an SGH organ label is admitted only when `Reviewed=Y`, `Q1=Y`, and `Q4=Y`, followed by technical QC. Unreviewed model predictions are not treated as ground truth.
- The locked test split is not used for model selection.
- `model_last.pth.tar` is updated every epoch for recovery. A better validation Dice updates `model_best.pth.tar` and deployable `model.ts`. Epochs 50/100/150/200 also get permanent checkpoint and TorchScript snapshots under the run's `checkpoints/` directory.


In [ ]:
from pathlib import Path
import os
import csv
import json
import shlex
import subprocess

PROJECT = Path.cwd().resolve()
HIVE_RUN = Path(os.environ.get(
    'PROSTATE_HIVE_RUN',
    '/home/ddsadmin/Documents/Prostate HIVE Inference 4 jun 2026 - highb fix',
)).expanduser().resolve()
TRAINING_IMAGE = 'prostate-mri-training:latest'
GPU_ID = 0

HELPER = PROJECT / 'scripts' / 'hive_training.py'
assert HELPER.is_file(), f'Open this notebook with {PROJECT} as the working directory'

BASE = [
    'python3', str(HELPER),
    '--project', str(PROJECT),
    '--hive-run', str(HIVE_RUN),
    '--image', TRAINING_IMAGE,
    '--gpu-id', str(GPU_ID),
]

def run_helper(*arguments):
    command = [*BASE, *map(str, arguments)]
    print('$', shlex.join(command), flush=True)
    subprocess.run(command, cwd=PROJECT, check=True)

print('Project:', PROJECT)
print('SGH source:', HIVE_RUN / 'output')
print('Transfer weights:', HIVE_RUN / 'models')
print('Physical GPU:', GPU_ID)


## 1. Mandatory preflight

Run this before starting or resuming training. It checks the image, exact source paths, all seven weight files, strict architecture compatibility, an idle physical GPU 0, and exactly one visible GPU inside the container.

In [ ]:
run_helper('preflight')

## 2. Prepare/rebuild the audited cohort

The cohort has already been prepared once in this workspace. Re-run this cell only after `audit_assignments.csv`, HIVE outputs, or ProstateX files change. It is deterministic and writes reports, cleaned organ labels, and locked splits to `artifacts/training/`.

In [ ]:
RUN_PREPARATION = False

if RUN_PREPARATION:
    run_helper('prepare')
else:
    print('Preparation not re-run. Set RUN_PREPARATION=True only when source/audit data changed.')

## 3. Review the cohort before training

Do not start if the report is missing or the counts are surprising. Detailed reasons are in `artifacts/training/exclusions.csv`; the human-readable overview is `artifacts/training/audit_report.md`.

In [ ]:
report_path = PROJECT / 'artifacts' / 'training' / 'audit_report.json'
assert report_path.is_file(), 'Run preparation first.'
report = json.loads(report_path.read_text())
print(json.dumps({
    'generated_at_utc': report['generated_at_utc'],
    'assigned_sgh_cases': report['assigned_cases'],
    'manifest_cases': report['manifest_cases'],
    'organ_eligible': report['organ_eligible'],
    'lesion_eligible': report['lesion_eligible'],
}, indent=2))

with (PROJECT / 'artifacts/training/organ_splits.csv').open(newline='') as handle:
    split_rows = list(csv.DictReader(handle))
split_counts = {}
for row in split_rows:
    split_counts[row['split']] = split_counts.get(row['split'], 0) + 1
print('Organ split counts:', split_counts)
print('Review:', PROJECT / 'artifacts/training/audit_report.md')
print('Exclusions:', PROJECT / 'artifacts/training/exclusions.csv')

## 4. Start a new organ transfer-learning run

Set the switch to `True` only after the preflight and cohort review pass. This is a long-running foreground cell; leave the kernel and terminal open. It initializes strictly from the existing deployed organ model.

The repository is mounted live at `/workspace`, so fixes to `training/*.py` and YAML files on this HIVE machine are used immediately by a newly started container; rebuilding `training_image.tar` is unnecessary here. If a run fails before `model_last.pth.tar` exists, correct the cause and start a new run. Resume only when a real last checkpoint exists.

In [ ]:
START_NEW_TRAINING = False
if START_NEW_TRAINING:
    run_helper('organ')
else:
    print('Not started. The completed organ run is preserved; set this True only for an intentional new run.')

## 5. Resume an interrupted organ run

Use the most recent `model_last.pth.tar` from the interrupted run. Resume restores the model, optimizer, scheduler, AMP scaler, epoch, and best metric. Do not use the new-run cell as well. If the failed run has no `model_last.pth.tar`, leave `RESUME_CHECKPOINT` blank and use the new-run cell instead.

In [ ]:
RESUME_CHECKPOINT = ''  # Example: experiments/organ/<run>/model_last.pth.tar

if RESUME_CHECKPOINT:
    resume_path = (PROJECT / RESUME_CHECKPOINT).resolve()
    run_helper('organ', '--resume', resume_path)
else:
    print('No resume checkpoint selected.')

## 6. Track runs and metrics

This works during or after training. `complete=no` means the run is still active or stopped before the final epoch; its `model_last.pth.tar` remains the recovery checkpoint.

In [ ]:
run_helper('status')

In [ ]:
runs = sorted((PROJECT / 'experiments/organ').glob('*/metrics.csv'))
if not runs:
    print('No organ metrics yet.')
else:
    metrics_path = runs[-1]
    with metrics_path.open(newline='') as handle:
        metrics = list(csv.DictReader(handle))
    scored = [row for row in metrics if row.get('val_dice')]
    best = max(scored, key=lambda row: float(row['val_dice'])) if scored else None
    print('Latest run:', metrics_path.parent)
    print('Last epoch:', metrics[-1]['epoch'] if metrics else 'none')
    print('Best validation row:', best)
    print('Recovery checkpoint:', metrics_path.parent / 'model_last.pth.tar')
    print('Best deployable model:', metrics_path.parent / 'model.ts')
    print('Permanent snapshots:', sorted((metrics_path.parent / 'checkpoints').glob('*')))

## 7. Next step

After the organ run is complete, train lesion folds 0–4 sequentially with `hive_train_lesion.ipynb`. That notebook packages the highest-validation completed organ run and all five highest-validation lesion runs into a checksum-verified `training_exports/deployment_<timestamp>/` directory ready to copy to the main PC or CARPL platform PC. Do not evaluate the locked test set until all model choices are frozen.